In [ ]:
# 3D array assignment SIMD-optimized with @avx macro
# Mac Studio
# 2025-03-27 
import Pkg

c = ["LoopVectorization", "Chain"]
Pkg.add(c)
using LoopVectorization

using Metal, Chain, BenchmarkTools

function main()
    # num_points = 200 * 200
    # num_edges = 176
    # div_n = 450
    num_points = 20 * 20
    num_edges = 200
    div_n = 450
    # div_n = 900
    # [点, 角度, 辺]
    cd_x = Array{Float32}(undef, num_points, div_n, num_edges)
    cd_y = Array{Float32}(undef, num_points, div_n, num_edges)
    ac_x = Array{Float32}(undef, num_points, div_n, num_edges)
    ac_y = Array{Float32}(undef, num_points, div_n, num_edges)

    cos_k0 =
    @chain collect(1:num_edges) begin
        (_ .- 1) .* (2 * π / num_edges)
        cos.()
        convert.(Float32, _)
    end

    sin_k0 =
    @chain collect(1:num_edges) begin
        (_ .- 1) .* (2 * π / num_edges)
        sin.()
        convert.(Float32, _)
    end

    cos_k1 =
    @chain collect(1:num_edges) begin
        (_) .* (2 * π / num_edges)
        cos.()
        convert.(Float32, _)
    end

    sin_k1 =
    @chain collect(1:num_edges) begin
        (_) .* (2 * π / num_edges)
        sin.()
        convert.(Float32, _)
    end

    cos_i =
    @chain collect(1:num_points) begin
        (_ .- 1) .* (2 * π / num_points)
        cos.()
        convert.(Float32, _)
    end

    sin_i =
    @chain collect(1:num_points) begin
        (_ .- 1) .* (2 * π / num_points)
        sin.()
        convert.(Float32, _)
    end

    @avx for k in 1:num_edges
        for j in 1:div_n
            for i in 1:num_points
                cd_x[i, j, k] = cos_k1[k] - cos_k0[k] # OK
                cd_y[i, j, k] = sin_k1[k] - sin_k0[k]
                ac_x[i, j, k] = cos_k0[k] - cos_i[i]
                ac_y[i, j, k] = sin_k0[k] - sin_i[i]
            end
        end
    end
end

@benchmark main()

In [1]:
# 3D Mtl array assignment via kernel function
# Mac Studio
# 2025-03-27
import Pkg
c = ["Metal", "Chain", "BenchmarkTools"]
Pkg.add(c)

using Metal, Chain, BenchmarkTools


# kernel function assignment
function kernel_assign(cd_x, cd_y, ac_x, ac_y, cos_k0, cos_k1, sin_k0, sin_k1, cos_i, sin_i)
    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i > size(cd_x, 1) || j > size(cd_x, 2) || k > size(cd_x, 3)        
        return
    end

    cd_x[i, j, k] = cos_k1[k] - cos_k0[k]
    cd_y[i, j, k] = sin_k1[k] - sin_k0[k]
    ac_x[i, j, k] = cos_k0[k] - cos_i[i]
    ac_y[i, j, k] = sin_k0[k] - sin_i[i]
    
    return nothing
end

function main()
    # num_points = 300 * 300
    # num_points = 200 * 200
    # num_edges = 176

    # α = Float32(0.5)
    # div_n = 
    # @chain Metal.current_device() begin
    #     _.maxBufferLength
    #     floor(α * (_ / (sizeof(Float32) * num_points * num_edges)))
    #     convert(Int, _)
    # end

    # num_points = 200 * 200
    # num_edges = 176
    # div_n = 450
    num_points = 20 * 20
    num_edges = 200
    div_n = 450
# div_n = 900
    cos_k0 = MtlArray{Float32}(undef, num_edges)
    sin_k0 = MtlArray{Float32}(undef, num_edges)
    cos_k1 = MtlArray{Float32}(undef, num_edges)
    sin_k1 = MtlArray{Float32}(undef, num_edges)
    cos_i = MtlArray{Float32}(undef, num_points)
    sin_i = MtlArray{Float32}(undef, num_points)

    cos_k0 =
    @chain collect(1:num_edges) begin
        (_ .- 1) .* (2 * π / num_edges)
        cos.()
        convert.(Float32, _)
        MtlArray(_)
    end
    
    sin_k0 =
    @chain collect(1:num_edges) begin
        (_ .- 1) .* (2 * π / num_edges)
        sin.()
        convert.(Float32, _)
        MtlArray(_)
    end
    
    cos_k1 =
    @chain collect(1:num_edges) begin
        (_) .* (2 * π / num_edges)
        cos.()
        convert.(Float32, _)
        MtlArray(_)
    end
    
    sin_k1 =
    @chain collect(1:num_edges) begin
        (_) .* (2 * π / num_edges)
        sin.()
        convert.(Float32, _)
        MtlArray(_)
    end

    cos_i =
    @chain collect(1:num_points) begin
        (_ .- 1) .* (2 * π / num_points)
        cos.()
        convert.(Float32, _)
        MtlArray(_)
    end
    
    sin_i =
    @chain collect(1:num_points) begin
        (_ .- 1) .* (2 * π / num_points)
        sin.()
        convert.(Float32, _)
        MtlArray(_)
    end
    
    mtl_cd_x = MtlArray{Float32}(undef, num_points, div_n, num_edges)
    mtl_cd_y = MtlArray{Float32}(undef, num_points, div_n, num_edges)
    mtl_ac_x = MtlArray{Float32}(undef, num_points, div_n, num_edges)
    mtl_ac_y = MtlArray{Float32}(undef, num_points, div_n, num_edges)

    # Threadgroup size configuration
    # threads_per_group = (10, 10, 10)  # MacPro M4MAX OK Mac Studio NG
    threads_per_group = (10, 9, 9)  # = 810 < 832
    num_groups = (ceil(Int, num_points / threads_per_group[1]), 
                ceil(Int, div_n / threads_per_group[2]), 
                ceil(Int, num_edges / threads_per_group[3]))
    
    # kernel function execution
    @metal threads = threads_per_group groups = num_groups kernel_assign(mtl_cd_x, mtl_cd_y, mtl_ac_x, mtl_ac_y,
    cos_k0, cos_k1, sin_k0, sin_k1, cos_i, sin_i)
 
    return

end

@benchmark main()


   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


BenchmarkTools.Trial: 119 samples with 1 evaluation per sample.
 Range (min … max):   5.193 ms … 138.302 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     19.459 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   42.645 ms ±  36.945 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

       ██▄                                                      
  ▅▁▁▅▅███▆▅▁▁▁▁▁▁▁▅▁▅▁▁▁▁▅▁▁▁▆▁▅▅▅▅▅▆▅▆▅██▅█▅▆▅▅▁▅▁▆▁▁▁▁█▁▁▁▅ ▅
  5.19 ms       Histogram: log(frequency) by time       135 ms <

 Memory estimate: 82.88 KiB, allocs estimate: 1282.

In [ ]:
# 3D array assignment using multithreading with @threads macro and 
#                           vectorization with @. macro
# Mac Studio
# Threads = 12
# 2025-03-27
import Pkg

c = ["Chain", "BenchmarkTools"]
Pkg.add(c)

using Base.Threads ,Chain, BenchmarkTools

# println("Threads.nthreads() = ", Threads.nthreads())
function main()
    # num_points = 200 * 200
    # num_edges = 176
    # div_n = 450
    num_points = 20 * 20
    num_edges = 200
    div_n = 450
    # [点, 角度, 辺]
    cd_x = Array{Float32}(undef, num_points, div_n, num_edges)
    cd_y = Array{Float32}(undef, num_points, div_n, num_edges)
    ac_x = Array{Float32}(undef, num_points, div_n, num_edges)
    ac_y = Array{Float32}(undef, num_points, div_n, num_edges)

    cos_k0 =
    @chain collect(1:num_edges) begin
        (_ .- 1) .* (2 * π / num_edges)
        cos.()
        convert.(Float32, _)
    end

    sin_k0 =
    @chain collect(1:num_edges) begin
        (_ .- 1) .* (2 * π / num_edges)
        sin.()
        convert.(Float32, _)
    end

    cos_k1 =
    @chain collect(1:num_edges) begin
        (_) .* (2 * π / num_edges)
        cos.()
        convert.(Float32, _)
    end

    sin_k1 =
    @chain collect(1:num_edges) begin
        (_) .* (2 * π / num_edges)
        sin.()
        convert.(Float32, _)
    end

    cos_i =
    @chain collect(1:num_points) begin
        (_ .- 1) .* (2 * π / num_points)
        cos.()
        convert.(Float32, _)
    end

    sin_i =
    @chain collect(1:num_points) begin
        (_ .- 1) .* (2 * π / num_points)
        sin.()
        convert.(Float32, _)
    end

    @threads for k in 1:num_edges
        @. cd_x[:, :, k] = cos_k1[k] - cos_k0[k]
        @. cd_y[:, :, k] = sin_k1[k] - sin_k0[k]
    end

    @threads for k in 1:num_edges
        for i in 1:num_points
            @. ac_x[i, :, k] = cos_k0[k] - cos_i[i]
            @. ac_y[i, :, k] = sin_k0[k] - sin_i[i]
        end
    end

end

@benchmark main()

In [ ]:
# @threadsマクロを使ったマルチスレッドによる並列化
import Pkg

c = ["LoopVectorization", "Chain", "BenchmarkTools"]
Pkg.add(c)

using LoopVectorization, Base.Threads, Chain, BenchmarkTools

# println("Threads.nthreads() = ", Threads.nthreads())
function main()
    # num_points = 200 * 200
    # num_edges = 176
    # div_n = 450
    num_points = 20 * 20
    num_edges = 200
    div_n = 450
    # [点, 角度, 辺]
    cd_x = Array{Float32}(undef, num_points, div_n, num_edges)
    cd_y = Array{Float32}(undef, num_points, div_n, num_edges)
    ac_x = Array{Float32}(undef, num_points, div_n, num_edges)
    ac_y = Array{Float32}(undef, num_points, div_n, num_edges)

    cos_k0 =
    @chain collect(1:num_edges) begin
        (_ .- 1) .* (2 * π / num_edges)
        cos.()
        convert.(Float32, _)
    end

    sin_k0 =
    @chain collect(1:num_edges) begin
        (_ .- 1) .* (2 * π / num_edges)
        sin.()
        convert.(Float32, _)
    end

    cos_k1 =
    @chain collect(1:num_edges) begin
        (_) .* (2 * π / num_edges)
        cos.()
        convert.(Float32, _)
    end

    sin_k1 =
    @chain collect(1:num_edges) begin
        (_) .* (2 * π / num_edges)
        sin.()
        convert.(Float32, _)
    end

    cos_i =
    @chain collect(1:num_points) begin
        (_ .- 1) .* (2 * π / num_points)
        cos.()
        convert.(Float32, _)
    end

    sin_i =
    @chain collect(1:num_points) begin
        (_ .- 1) .* (2 * π / num_points)
        sin.()
        convert.(Float32, _)
    end

    @threads for k in 1:num_edges
        for j in 1:div_n
            for i in 1:num_points
                cd_x[i, j, k] = cos_k1[k] - cos_k0[k] # OK
                cd_y[i, j, k] = sin_k1[k] - sin_k0[k]
                ac_x[i, j, k] = cos_k0[k] - cos_i[i]
                ac_y[i, j, k] = sin_k0[k] - sin_i[i]
            end
        end
    end

end

@benchmark main()

In [ ]:
# Allocation of isomorphic 3D Mtl array memory(using similar function)
# Mac Studio
# 2025-03-27
import Pkg
c = ["Metal", "BenchmarkTools"]
Pkg.add(c)

using Metal, BenchmarkTools

# Using similar function
function code1(n_x, n_y, n_e)
    Mtl_vec0_x = MtlArray{Float32}(undef, n_x, n_y, n_e)
    Mtl_vec0_y = similar(Mtl_vec0_x)
    Mtl_vec1_x = similar(Mtl_vec0_x)
    Mtl_vec1_y = similar(Mtl_vec0_x)

end

n_x = 100
n_y = 100
n_e = 100

@benchmark code1(n_x, n_y, n_e)


In [ ]:
# Allocation of isomorphic 3D Mtl array memory(without using similar function)
# Mac Studio
# 2025-03-27
import Pkg
c = ["Metal", "BenchmarkTools"]
Pkg.add(c)

using Metal, BenchmarkTools

# Without using similar
function code2(n_x, n_y, n_e)
    Mtl_vec0_x = MtlArray{Float32}(undef, n_x, n_y, n_e)
    Mtl_vec0_y = MtlArray{Float32}(undef, n_x, n_y, n_e)
    Mtl_vec1_x = MtlArray{Float32}(undef, n_x, n_y, n_e)
    Mtl_vec1_y = MtlArray{Float32}(undef, n_x, n_y, n_e)
    
end

n_x = 100
n_y = 100
n_e = 100

@benchmark code2(n_x, n_y, n_e)

In [ ]:
# Vectorization using repeat function and reshape function
# Mac Studio
# 2025-03-28
import Pkg
c = ["DataFrames", "Tidier", "Chain", "BenchmarkTools"]
Pkg.add(c)
using DataFrames, Tidier, Chain, BenchmarkTools

function in_out_judge(xx::Vector{Float32}, yy::Vector{Float32}, poly_edge::DataFrame)
    n_x, n_y = length(xx), length(yy)
    poly_ids = unique(poly_edge.id)

    for (p_idx, p) in enumerate(poly_ids)
        edge =
        @chain poly_edge begin
            @filter(id == !!p)
            @select(x0, y0, x1, y1)
            # @filter(:id => ==(p))
            # @select(:x0, :y0, :x1, :y1)
        end

        n_e = nrow(edge)
        winding_angle = zeros(Float32, n_x, n_y)

        vec0_x = repeat(edge.x0, outer = (1, n_x, n_y)) .- reshape(xx, (1, n_x, 1))
        vec0_y = repeat(edge.y0, outer = (1, n_x, n_y)) .- reshape(yy, (1, 1, n_y))
        vec1_x = repeat(edge.x1, outer = (1, n_x, n_y)) .- reshape(xx, (1, n_x, 1))
        vec1_y = repeat(edge.y1, outer = (1, n_x, n_y)) .- reshape(yy, (1, 1, n_y))

    end

end

n_size = 200
xx, yy = rand(Float32, n_size), rand(Float32, n_size)
poly_edge = DataFrame(id = rand(1:10, n_size), x0 = rand(Float32, n_size), y0 = rand(Float32, n_size), x1 = rand(Float32, n_size), y1 = rand(Float32, n_size))
@benchmark in_out_judge(xx, yy, poly_edge)


In [ ]:
# Using Metal arrays within a for-loop
# Mac Studio
# 2025-03-29
import Pkg
c = ["DataFrames", "Tidier", "Chain", "BenchmarkTools", "Metal"]
Pkg.add(c)
using DataFrames, Tidier, Chain, BenchmarkTools, Metal

function in_out_judge(xx::Vector{Float32}, yy::Vector{Float32}, poly_edge::DataFrame)
    n_x, n_y = length(xx), length(yy)
    poly_ids = unique(poly_edge.id)

    for (p_idx, p) in enumerate(poly_ids)
        edge =
        @chain poly_edge begin
            @filter(id == !!p)
            @select(x0, y0, x1, y1)
            # @filter(:id => ==(p))
            # @select(:x0, :y0, :x1, :y1)
        end

        n_e = nrow(edge)
        winding_angle = zeros(Float32, n_x, n_y)

        Mtl_vec0_x = MtlArray{Float32}(undef, n_x, n_y, n_e)
        Mtl_vec0_y = similar(Mtl_vec0_x)
        Mtl_vec1_x = similar(Mtl_vec0_x)
        Mtl_vec1_y = similar(Mtl_vec0_x)

        for j in 1:n_e
            for i in 1:n_x
                Mtl_vec0_x[i, :, j] .= edge.x0[j] .- xx[i]
                Mtl_vec1_x[i, :, j] .= edge.x1[j] .- xx[i]
            end
            for i in 1:n_y
                Mtl_vec0_y[:, i, j] .= edge.y0[j] .- yy[i]
                Mtl_vec1_y[:, i, j] .= edge.y1[j] .- yy[i]
            end
        end
    end

end

n_size = 200
xx, yy = rand(Float32, n_size), rand(Float32, n_size)
poly_edge = DataFrame(id = rand(1:10, n_size), x0 = rand(Float32, n_size), y0 = rand(Float32, n_size), x1 = rand(Float32, n_size), y1 = rand(Float32, n_size))
@benchmark in_out_judge(xx, yy, poly_edge)

In [ ]:
# 3D array assignment using Metal arrays
# Mac Studio
# 2025-03-29
import Pkg
c = ["DataFrames", "Tidier", "Chain", "BenchmarkTools", "Metal"]
Pkg.add(c)
using DataFrames, Tidier, Chain, BenchmarkTools, Metal

# Kernel function assignment
function kernel_assign_0(Mtl_vec0_x::MtlDeviceArray{Float32, 3}, Mtl_vec0_y::MtlDeviceArray{Float32, 3}, 
                            Mtl_vec1_x::MtlDeviceArray{Float32, 3}, Mtl_vec1_y::MtlDeviceArray{Float32, 3},
                            xx::MtlDeviceArray{Float32, 1}, yy::MtlDeviceArray{Float32, 1}, 
                            x0::MtlDeviceArray{Float32, 1}, y0::MtlDeviceArray{Float32, 1},
                            x1::MtlDeviceArray{Float32, 1}, y1::MtlDeviceArray{Float32, 1})
    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i > size(Mtl_vec0_x, 1) || j > size(Mtl_vec0_x, 2) || k > size(Mtl_vec0_x, 3)
        return nothing
    end

    Mtl_vec0_x[i, j, k] = x0[k] - xx[i]
    Mtl_vec1_x[i, j, k] = x1[k] - xx[i]
    Mtl_vec0_y[i, j, k] = y0[k] - yy[j]
    Mtl_vec1_y[i, j, k] = y1[k] - yy[j]

    return nothing

end

function in_out_judge(xx::Vector{Float32}, yy::Vector{Float32}, poly_edge::DataFrame)
    n_x, n_y = length(xx), length(yy)
    poly_ids = unique(poly_edge.id)

    for (p_idx, p) in enumerate(poly_ids)
        edge =
        @chain poly_edge begin
            @filter(id == !!p)
            @select(x0, y0, x1, y1)
        end

        Mtl_x0 = MtlArray{Float32}(edge.x0)
        Mtl_y0 = MtlArray{Float32}(edge.y0)
        Mtl_x1 = MtlArray{Float32}(edge.x1)
        Mtl_y1 = MtlArray{Float32}(edge.y1)

        Mtl_xx = MtlArray{Float32}(xx)
        Mtl_yy = MtlArray{Float32}(yy)

        n_e = nrow(edge)
        winding_angle = zeros(Float32, n_x, n_y)


        Mtl_vec0_x = MtlArray{Float32}(undef, n_x, n_y, n_e)
        Mtl_vec0_y = similar(Mtl_vec0_x)
        Mtl_vec1_x = similar(Mtl_vec0_x)
        Mtl_vec1_y = similar(Mtl_vec0_x)
        # Threadgroup size configuration
        threads_per_group = (10, 9, 9)  # = 810 < 832
        num_groups = (ceil(Int, n_x / threads_per_group[1]), 
                    ceil(Int, n_y / threads_per_group[2]), 
                    ceil(Int, n_e / threads_per_group[3]))
        # Kernel function execution
        @metal threads = threads_per_group groups = num_groups kernel_assign_0(Mtl_vec0_x, Mtl_vec0_y, 
                                                                                Mtl_vec1_x, Mtl_vec1_y, 
                                                                                Mtl_xx, Mtl_yy, 
                                                                                Mtl_x0, Mtl_y0, Mtl_x1, Mtl_y1)
    end
end

n_size = 200
xx, yy = rand(Float32, n_size), rand(Float32, n_size)
poly_edge = DataFrame(id = rand(1:10, n_size), x0 = rand(Float32, n_size), y0 = rand(Float32, n_size), x1 = rand(Float32, n_size), y1 = rand(Float32, n_size))
@benchmark in_out_judge(xx, yy, poly_edge)


In [ ]:
# Implementing for-loop as a kernel function with Metal
# Mac Studio
# 2025-03-29
import Pkg
c = ["DataFrames", "Tidier", "Chain", "BenchmarkTools", "Metal"]
Pkg.add(c)
using DataFrames, Tidier, Chain, BenchmarkTools, Metal

function kernel_assign_vector(xx, yy, edge, n_x, n_y, n_e)
    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i > n_x || j > n_y || k > n_e
        return
    end

    xx[i] = edge.x0[k] .- xx[i]
    yy[j] = edge.y0[k] .- yy[j]

    return nothing
end

function in_out_judge(xx::Vector{Float32}, yy::Vector{Float32}, poly_edge::DataFrame)
    n_x, n_y = length(xx), length(yy)
    poly_ids = unique(poly_edge.id)

    for (p_idx, p) in enumerate(poly_ids)
        edge =
        @chain poly_edge begin
            @filter(id == !!p)
            @select(x0, y0, x1, y1)
            # @filter(:id => ==(p))
            # @select(:x0, :y0, :x1, :y1)
        end

        n_e = nrow(edge)
        winding_angle = zeros(Float32, n_x, n_y)

        Mtl_vec0_x = MtlArray{Float32}(undef, n_x, n_y, n_e)
        Mtl_vec0_y = similar(Mtl_vec0_x)
        Mtl_vec1_x = similar(Mtl_vec0_x)
        Mtl_vec1_y = similar(Mtl_vec0_x)

        for j in 1:n_e
            for i in 1:n_x
                Mtl_vec0_x[i, :, j] .= edge.x0[j] .- xx[i]
                Mtl_vec1_x[i, :, j] .= edge.x1[j] .- xx[i]
            end
            for i in 1:n_y
                Mtl_vec0_y[:, i, j] .= edge.y0[j] .- yy[i]
                Mtl_vec1_y[:, i, j] .= edge.y1[j] .- yy[i]
            end
        end
    end

end

n_size = 200
xx, yy = rand(Float32, n_size), rand(Float32, n_size)
poly_edge = DataFrame(id = rand(1:10, n_size), x0 = rand(Float32, n_size), y0 = rand(Float32, n_size), x1 = rand(Float32, n_size), y1 = rand(Float32, n_size))
@benchmark in_out_judge(xx, yy, poly_edge)

In [ ]:
# Kernel function-based validity check
# Mac Studio
# 2025-03-31
import Pkg
c = ["DataFrames", "Tidier", "Chain", "BenchmarkTools", "Metal"]
Pkg.add(c)
using DataFrames, Tidier, Chain, BenchmarkTools, Metal

function kernel_validity(Mtl_s::MtlDeviceArray{Float32, 3}, Mtl_t::MtlDeviceArray{Float32, 3}, 
                            Mtl_s_valid::MtlDeviceArray{Bool, 3}, Mtl_t_valid::MtlDeviceArray{Bool, 3})::Nothing

    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i > size(Mtl_s, 1) || j > size(Mtl_s, 2) || k > size(Mtl_s, 3)
        return nothing
    end

    Mtl_s_valid[i, j, k] = (Mtl_s[i, j, k] >= 0 && Mtl_s[i, j, k] <= 1)
    Mtl_t_valid[i, j, k] = (Mtl_t[i, j, k] >= 0 && Mtl_t[i, j, k] <= 1)

    return nothing

end

function main(Mtl_s, Mtl_t)
    num_points = 200 * 200
    div_n = 450
    num_edges = 176

    Mtl_s = MtlArray{Float32}(undef, num_points, div_n, num_edges)
    Mtl_t = similar(Mtl_s)
    Mtl_s_valid = similar(Mtl_s, Bool)
    Mtl_t_valid = similar(Mtl_s, Bool)

    Mtl_s .= rand(Float32, num_points, div_n, num_edges)
    Mtl_t .= rand(Float32, num_points, div_n, num_edges)

    # Threadgroup size configuration
    threads_per_group = (10, 9, 9)  # = 810 < 832
    num_groups = (ceil(Int, num_points / threads_per_group[1]), 
                ceil(Int, div_n / threads_per_group[2]), 
                ceil(Int, num_edges / threads_per_group[3]))
    @metal threads = threads_per_group groups = num_groups kernel_validity(Mtl_s, Mtl_t, Mtl_s_valid, Mtl_t_valid)

end

@benchmark main()

In [ ]:
# Sift row elements of 2D array back by one column using a for loop
# Mac Studio
# 2025-04-01

import Pkg
c = ["DataFrames", "Tidier", "Chain", "BenchmarkTools", "Metal"]
Pkg.add(c)
using DataFrames, Tidier, Chain, BenchmarkTools, Metal

function shift_row_elements(Mtl_crossing_x::MtlArray{Float32, 2}, num_points::Int, div_n::Int)
    crossing_x = Array(Mtl_crossing_x)

    crossing_next_x = similar(crossing_x)
    for j in 1:div_n
        for i in 1:num_points
            crossing_next_x[i, j] = crossing_x[i, mod1(j + 1, div_n)]
        end
    end
    Mtl_crossing_next_x = MtlArray{Float32}(crossing_next_x)

end

const div_v = 450
const num_points = 200 * 200

tmp = rand(Float32, num_points, div_v)
Mtl_crossing_x = MtlArray{Float32}(tmp)

@benchmark shift_row_elements(Mtl_crossing_x, num_points, div_v)

In [ ]:
# Sift row elements of 2D array back by one column using a vectorization for loop
# Mac Studio
# 2025-04-01

import Pkg
c = ["DataFrames", "Tidier", "Chain", "BenchmarkTools", "Metal", "LoopVectorization"]
Pkg.add(c)
using DataFrames, Tidier, Chain, BenchmarkTools, Metal, LoopVectorization

function shift_row_elements(Mtl_crossing_x::MtlArray{Float32, 2}, num_points::Int, div_n::Int)
    crossing_x = Array(Mtl_crossing_x)

    crossing_next_x = similar(crossing_x)
    @avx for j in 1:div_n
        for i in 1:num_points
            crossing_next_x[i, j] = crossing_x[i, mod1(j + 1, div_n)]
        end
    end
    Mtl_crossing_next_x = MtlArray{Float32}(crossing_next_x)

end

const div_v = 450
const num_points = 200 * 200

tmp = rand(Float32, num_points, div_v)
Mtl_crossing_x = MtlArray{Float32}(tmp)

@benchmark shift_row_elements(Mtl_crossing_x, num_points, div_v)

In [ ]:
# Sift row elements of 2D array back by one column using a kernel function
# Mac Studio
# 2025-04-01

import Pkg
c = ["DataFrames", "Tidier", "Chain", "BenchmarkTools", "Metal"]
Pkg.add(c)
using DataFrames, Tidier, Chain, BenchmarkTools, Metal

function kernel_shift(crossing_next_x::MtlDeviceArray{Float32, 2}, crossing_x::MtlDeviceArray{Float32, 2}, num_points::Int, div_n::Int):Nothing

    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y

    if i > num_points || j > div_n
        return nothing
    end

    crossing_next_x[i, j] = crossing_x[i, mod1(j + 1, div_n)]

    return nothing

end

function shift_row_elements(Mtl_crossing_x::MtlArray{Float32, 2}, num_points::Int, div_n::Int)

    threads_per_group = (32, 32)
    num_groups = (ceil(Int, num_points / threads_per_group[1]), ceil(Int, div_n / threads_per_group[2]))
    Mtl_crossing_next_x = similar(Mtl_crossing_x)
    @metal threads = threads_per_group groups = num_groups kernel_shift(Mtl_crossing_next_x, Mtl_crossing_x, num_points, div_n)
 
end

const div_v = 450
const num_points = 200 * 200

tmp = rand(Float32, num_points, div_v)
Mtl_crossing_x = MtlArray{Float32}(tmp)

@benchmark shift_row_elements(Mtl_crossing_x, num_points, div_v)

In [ ]:
# 座標法による多角形面積計算
import Pkg
c = ["DataFrames", "Metal", "Chain", "BenchmarkTools"]
Pkg.add(c)

using DataFrames, Metal, Chain, BenchmarkTools

# カーネル関数を使った多角形の面積計算（座標法）
function kernel_shoelace(isovist_partial::MtlDeviceArray{Float32, 2}, A_Index_partial::MtlDeviceArray{Float32, 2},
                            crossing_x::MtlDeviceArray{Float32, 2}, crossing_y::MtlDeviceArray{Float32, 2},
                            adj_x::MtlDeviceArray{Float32, 2}, adj_y::MtlDeviceArray{Float32, 2})::Nothing

    i = thread_position_in_grid_2d().x
    j = thread_position_in_grid_2d().y

    if i > size(isovist_partial, 1) || j > size(isovist_partial, 2)
        return nothing
    end

    isovist_partial[i, j] = (crossing_x[i, j] * crossing_y[i, mod1(j + 1, size(crossing_y, 2))] - 
                                crossing_y[i, j] * crossing_x[i, mod1(j + 1, size(crossing_x, 2))]) / 2.0f0
    A_Index_partial[i, j] = (adj_x[i, j] * adj_y[i, mod1(j + 1, size(adj_y, 2))] - 
                                adj_y[i, j] * adj_x[i, mod1(j + 1, size(adj_x, 2))]) / 2.0f0

    return nothing

end

function main()
    crossing_x = [0.0 10.0 10.0 5.0 0.0;
                  0.0 10.0 10.0 5.0 0.0]
    crossing_y = [0.0 0.0 10.0 5.0 10.0;
                  0.0 0.0 10.0 20.0 10.0]

    Mtl_crossing_x = MtlArray{Float32}(crossing_x)
    Mtl_crossing_y = MtlArray{Float32}(crossing_y)
    Mtl_isovist_partial = similar(Mtl_crossing_x)

    adj_x = [0.0 10.0 10.0 5.0 0.0;
                  0.0 10.0 10.0 5.0 0.0]
    adj_y = [0.0 0.0 10.0 5.0 10.0;
                  0.0 0.0 10.0 20.0 10.0]

    Mtl_adj_x = MtlArray{Float32}(adj_x)
    Mtl_adj_y = MtlArray{Float32}(adj_y)
    Mtl_A_Index_partial = similar(Mtl_adj_x)

    num_points = size(crossing_x, 1)
    div_n = size(crossing_x, 2)

    threads_per_group = (27, 27)  # = 729 < 768
    num_groups = (ceil(Int, num_points / threads_per_group[1]), ceil(Int, div_n / threads_per_group[2]))

    @metal threads = threads_per_group groups = num_groups kernel_shoelace(Mtl_isovist_partial, Mtl_A_Index_partial,
                                                                            Mtl_crossing_x, Mtl_crossing_y,
                                                                            Mtl_adj_x, Mtl_adj_y)

    isovist = @chain Mtl_isovist_partial begin
        sum(_, dims = 2)
        reshape(_, num_points, 1)
    end

    A_Index_partial = @chain Mtl_A_Index_partial begin
        sum(_, dims = 2)
        reshape(_, num_points, 1)
    end

end

@benchmark main()

In [ ]:
# 座標法による多角形面積計算
import Pkg
c = ["DataFrames", "Metal", "Chain", "BenchmarkTools"]
Pkg.add(c)

using DataFrames, Metal, Chain, BenchmarkTools

# カーネル関数を使った多角形の面積計算（座標法）
function kernel_shoelace(partial::MtlDeviceArray{Float32, 2},
                            xx::MtlDeviceArray{Float32, 2}, yy::MtlDeviceArray{Float32, 2})::Nothing

    i = thread_position_in_grid_2d().x
    j = thread_position_in_grid_2d().y

    if i > size(partial, 1) || j > size(partial, 2)
        return nothing
    end

    partial[i, j] = (xx[i, j] * yy[i, mod1(j + 1, size(yy, 2))] - 
                                yy[i, j] * xx[i, mod1(j + 1, size(xx, 2))]) / 2.0f0

    return nothing

end

function main()
    crossing_x = [0.0 10.0 10.0 5.0 0.0;
                  0.0 10.0 10.0 5.0 0.0]
    crossing_y = [0.0 0.0 10.0 5.0 10.0;
                  0.0 0.0 10.0 20.0 10.0]

    Mtl_crossing_x = MtlArray{Float32}(crossing_x)
    Mtl_crossing_y = MtlArray{Float32}(crossing_y)
    Mtl_isovist_partial = similar(Mtl_crossing_x)

    adj_x = [0.0 10.0 10.0 5.0 0.0;
                  0.0 10.0 10.0 5.0 0.0]
    adj_y = [0.0 0.0 10.0 5.0 10.0;
                  0.0 0.0 10.0 20.0 10.0]

    Mtl_adj_x = MtlArray{Float32}(adj_x)
    Mtl_adj_y = MtlArray{Float32}(adj_y)
    Mtl_A_Index_partial = similar(Mtl_adj_x)

    num_points = size(crossing_x, 1)
    div_n = size(crossing_x, 2)

    threads_per_group = (27, 27)  # = 729 < 768
    num_groups = (ceil(Int, num_points / threads_per_group[1]), ceil(Int, div_n / threads_per_group[2]))

    @metal threads = threads_per_group groups = num_groups kernel_shoelace(Mtl_isovist_partial,
                                                                            Mtl_crossing_x, Mtl_crossing_y)

    @metal threads = threads_per_group groups = num_groups kernel_shoelace(Mtl_A_Index_partial,
                                                                            Mtl_adj_x, Mtl_adj_y)

    isovist = @chain Mtl_isovist_partial begin
        sum(_, dims = 2)
        reshape(_, num_points, 1)
    end
    
    A_Index = @chain Mtl_A_Index_partial begin
        sum(_, dims = 2)
        reshape(_, num_points, 1)
    end

end

@benchmark main()

In [5]:
import Pkg
Pkg.status()

Status `~/.julia/environments/v1.11/Project.toml`
  [6e4b80f9] BenchmarkTools v1.6.0
  [336ed68f] CSV v0.10.15
  [8be319e6] Chain v0.6.0
  [a93c6f00] DataFrames v1.7.0
  [31c24e10] Distributions v0.25.119
  [bdcacae8] LoopVectorization v0.12.172
  [dde4c033] Metal v1.5.1
  [438e738f] PyCall v1.96.4
  [1fd47b50] QuadGK v2.11.2
  [6f49c342] RCall v0.14.8
⌃ [f0413319] Tidier v1.5.1
Info Packages marked with ⌃ have new versions available and may be upgradable.


In [6]:
import Pkg
c = ["Tidier"]
Pkg.update(c)

using Tidier, LoopVectorization

    Updating registry at `~/.julia/registries/General.toml`
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [ ]:
import Pkg
c = ["Metal", "Tidier"]
Pkg.rm(c)
Pkg.add(c)